# Skin MRI Analysis (Synthetic DAD & NACRS) — Plotly Walkthrough

This notebook walks through the **Skin MRI** use-case end-to-end using **Plotly** for interactive charts.

**Prerequisites**
- Run the SQL pipeline first so the SQLite DB and views exist:
  ```bash
  make gen-data
  make sql
  ```
- Optional (if Plotly isn't installed):
  ```bash
  pip install plotly pandas
  ```

**What you'll see here**
1. Load data from SQLite views
2. Compute simple KPIs
3. Visualize with Plotly:
   - Top diagnoses (bar)
   - Length of stay by Skin MRI flag (box)
   - ER revisit timing (histogram)
   - KPI cards (indicators)
4. Export interactive HTML plots (optional)

In [1]:
# --- Setup & imports ---
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import os

# Plotly (interactive)
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


# Paths
ROOT = Path(os.getcwd()).parent.parent
DBPATH = ROOT / "data" / "db" / "health_admin_demo.sqlite"
OUT_T = ROOT / "outputs" / "tables"
OUT_F = ROOT / "outputs" / "figures"
OUT_I = OUT_F / "interactive"
OUT_T.mkdir(parents=True, exist_ok=True)
OUT_F.mkdir(parents=True, exist_ok=True)
OUT_I.mkdir(parents=True, exist_ok=True)

assert DBPATH.exists(), f"DB not found at {DBPATH}. Please run `make sql` first."
conn = sqlite3.connect(DBPATH)

In [2]:
# --- Load views into DataFrames ---
top_diag = pd.read_sql_query("SELECT * FROM v_skin_mri_top_diag;", conn)
los_by_flag = pd.read_sql_query("SELECT * FROM v_los_by_skin_mri;", conn)
revisit = pd.read_sql_query("SELECT * FROM v_skin_mri_er_revisit;", conn)
dad = pd.read_sql_query("SELECT EncID, SkinMRI_flag, LOS_calc FROM v_dad_clean;", conn)

# Basic sanity checks
display(top_diag.head())
display(los_by_flag.head())
display(revisit.head())
display(dad.head())

,DiagnosisCode,Count
0,N39.0,83
1,S72.0,82
2,F32.9,76
3,E11.9,74
4,M54.5,72


,SkinMRI_flag,AvgLOS,N
0,0,4.03,11208
1,1,3.94,792


,EncID,PatientID,DischargeDate,ER_VisitDateTime,delta_days
0,11847,3531,2024-08-17 03:01:43,2024-09-14 00:31:26,28
1,10070,3557,2024-11-02 00:08:14,2024-11-22 02:35:05,20
2,131,824,2022-08-11 07:30:49,2022-08-16 16:32:27,5
3,8264,3090,2024-06-26 08:22:19,2024-07-09 20:40:35,14
4,3518,102,2023-03-28 18:33:43,2023-04-02 11:38:06,5


,EncID,SkinMRI_flag,LOS_calc
0,1,0,6
1,2,0,5
2,3,1,9
3,4,0,5
4,5,0,4


In [3]:
# --- KPIs ---
n_skin = pd.read_sql_query("SELECT COUNT(*) AS n FROM v_skin_mri_dad;", conn).iloc[0,0]
n_dad  = pd.read_sql_query("SELECT COUNT(*) AS n FROM v_dad_clean;", conn).iloc[0,0]
share = round(100.0 * n_skin / n_dad, 2) if n_dad else 0.0

n_skin_revisit = revisit["EncID"].nunique() if not revisit.empty else 0
revisit_rate = round(100.0 * n_skin_revisit / n_skin, 2) if n_skin else 0.0

kpi_df = pd.DataFrame({
    "Metric": ["Total DAD Encounters", "Skin MRI Encounters", "Skin MRI Share (%)", "30d ER Revisit in Skin MRI (%)"],
    "Value": [n_dad, n_skin, share, revisit_rate]
})
kpi_df

,Metric,Value
0,Total DAD Encounters,12000.00
1,Skin MRI Encounters,792.00
2,Skin MRI Share (%),6.60
3,30d ER Revisit in Skin MRI (%),5.05


## 1) Top Diagnoses (MRDx) among Skin MRI encounters

Bar chart of the most frequent **MRDx** codes for Skin MRI encounters.

In [4]:
# Plotly horizontal bar
td = top_diag.sort_values("Count", ascending=True)
fig1 = px.bar(
    td,
    x="Count",
    y="DiagnosisCode",
    orientation="h",
    title="Top MRDx among Skin MRI Encounters"
)
fig1.update_layout(yaxis_title="ICD-10-CA Code", xaxis_title="Count")
fig1.show()

# Optional: save as interactive HTML
pio.write_html(fig1, file=str(OUT_I / "top_diagnoses_skin_mri.html"), auto_open=False, include_plotlyjs="cdn")

## 2) Length of Stay by Skin MRI flag (box plot)

We compare **LOS** distributions between encounters **with** Skin MRI vs **without**.

In [5]:
# Prepare tidy data for box plot
dad_plot = dad.copy()
dad_plot["SkinMRI"] = np.where(dad_plot["SkinMRI_flag"]==1, "Skin MRI", "Non-Skin MRI")

fig2 = px.box(
    dad_plot,
    x="SkinMRI",
    y="LOS_calc",
    points=False,
    title="Length of Stay by Skin MRI Flag"
)
fig2.update_layout(yaxis_title="Days", xaxis_title="Cohort")
fig2.show()

pio.write_html(fig2, file=str(OUT_I / "los_by_skin_mri.html"), auto_open=False, include_plotlyjs="cdn")

## 3) ER revisit timing (histogram)

Histogram of days from discharge to the first **ER revisit** (within 30 days) for the Skin MRI cohort.

In [6]:
if not revisit.empty:
    fig3 = px.histogram(
        revisit.assign(delta_days=revisit["delta_days"].astype(int)),
        x="delta_days",
        nbins=15,
        title="Distribution of ER Revisit Timing (Skin MRI cohort)"
    )
    fig3.update_layout(xaxis_title="Days from discharge", yaxis_title="Count of revisits")
    fig3.show()
    pio.write_html(fig3, file=str(OUT_I / "er_revisit_hist.html"), auto_open=False, include_plotlyjs="cdn")
else:
    print("No ER revisits found in the Skin MRI cohort for this synthetic sample.")

## 4) KPI cards (optional)

Use Plotly **Indicators** for a compact KPI panel.

In [7]:
fig_kpi = go.Figure()

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=n_dad,
    title={"text": "Total DAD Encounters"},
    domain={'x': [0.0, 0.24], 'y': [0, 1]}
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=n_skin,
    title={"text": "Skin MRI Encounters"},
    domain={'x': [0.26, 0.5], 'y': [0, 1]}
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=share,
    number={"suffix":"%"},
    title={"text": "Skin MRI Share"},
    domain={'x': [0.52, 0.76], 'y': [0, 1]}
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=revisit_rate,
    number={"suffix":"%"},
    title={"text": "30d ER Revisit (Skin MRI)"},
    domain={'x': [0.78, 1.0], 'y': [0, 1]}
))

fig_kpi.update_layout(title="KPI Panel")
fig_kpi.show()

pio.write_html(fig_kpi, file=str(OUT_I / "kpi_panel.html"), auto_open=False, include_plotlyjs="cdn")